학습 전 whisper 모델 성능 테스트 코드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install evaluate jiwer

In [ ]:
import torch
import librosa
import evaluate
from transformers import WhisperForConditionalGeneration, WhisperProcessor

print("Whisper(Base) 모델 다운로드 중...")

model_id = "openai/whisper-base"
processor = WhisperProcessor.from_pretrained(model_id, language="Korean", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.config.forced_decoder_ids = None

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)

print("준비 완료. 이제 음성 파일을 던져주세요.")

In [ ]:
import torch
import librosa
import evaluate
from transformers import WhisperForConditionalGeneration, WhisperProcessor

# 에러율 평가 지표 불러오기
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

model_id = "openai/whisper-base"
processor = WhisperProcessor.from_pretrained(model_id, language="Korean", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.config.forced_decoder_ids = None

device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)

print("채점 준비 완료")

In [ ]:
from google.colab import files
import os
from tqdm import tqdm

print("짝이 맞는 .wav 파일과 .txt 파일들을 한꺼번에 선택해 주세요")
uploaded = files.upload()

# 파일 확장자별로 분류
wav_files = sorted([f for f in uploaded.keys() if f.endswith('.wav')])
txt_files = sorted([f for f in uploaded.keys() if f.endswith('.txt')])

references = []   # 실제 정답 문장들
predictions = []  # AI가 받아쓴 문장들

print(f"\n총 {len(wav_files)}개의 썰린 파일로 즉석 Baseline 평가를 시작합니다...")

# 하나씩 듣고 받아쓰고 채점
for wav_name in tqdm(wav_files):
    # 짝이 맞는 txt 파일 이름 찾기
    base_name = os.path.splitext(wav_name)[0]
    txt_name = base_name + '.txt'
    
    if txt_name not in uploaded:
        print(f"{wav_name}과 짝이 맞는 텍스트 파일이 업로드되지 않아 스킵합니다.")
        continue
        
    # 정답 텍스트 읽기
    reference_text = uploaded[txt_name].decode('utf-8').strip()
    references.append(reference_text)
    
    # 오디오 읽고 AI 추론
    audio_array, sampling_rate = librosa.load(wav_name, sr=16000)
    input_features = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt").input_features.to(device)
    
    predicted_ids = model.generate(input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    predictions.append(transcription)

# 최종 에러율 계산
if len(references) > 0:
    wer_score = 100 * wer_metric.compute(predictions=predictions, references=references)
    cer_score = 100 * cer_metric.compute(predictions=predictions, references=references)

    print("\n" + "="*50)
    print("[학습 전 base Whisper 성적표]")
    print(f"단어 에러율 (WER): {wer_score:.2f}%")
    print(f"글자 에러율 (CER): {cer_score:.2f}%")
    print("="*50)


    print("\n[실제 오답 샘플]")
    for i in range(min(3, len(references))):
        print(f"[{i+1}] 실제 정답: {references[i]}")
        print(f"    AI 인식: {predictions[i]}")
else:
    print("업로드된 파일 중 짝이 맞는 wav와 txt 파일이 없습니다.")